In [ ]:
## N4 BIAS FIELD CORRECTION

import nibabel as nib
from ipywidgets import interact
from matplotlib import pyplot as plt
import numpy as np
import SimpleITK as sitk

In [ ]:
# Load Nifti File
nifti = "E:\\Documents\\N4_Test.nii"
#nifti = "E:\\Documents\\DipySegmentation\\LP-0001-01-01-01_NIFTI\\1_fl3d_ax_sel_10x10x10.nii.gz" #copy path to nifti file
image = nib.load(nifti)
raw_nifti = image

In [ ]:
# Transform image to Array
raw_img_sitk = sitk.ReadImage(nifti, sitk.sitkFloat32)
raw_img_sitk_arr = sitk.GetArrayFromImage(raw_img_sitk)

# Rescale and apply Threshold of correction
transformed = sitk.RescaleIntensity(raw_img_sitk, 0, 255)
transformed = sitk.LiThreshold(transformed,0,1)
head_mask = transformed

# Shrink image (may need to adjust this setting)
shrinkFactor = 4
inputImage = raw_img_sitk
inputImage = sitk.Shrink( raw_img_sitk, [ shrinkFactor ] * inputImage.GetDimension() )
maskImage = sitk.Shrink( head_mask, [ shrinkFactor ] * inputImage.GetDimension() ) 

# Obtain correcter
bias_corrector = sitk.N4BiasFieldCorrectionImageFilter()

# Get final corrected image
corrected = bias_corrector.Execute(inputImage, maskImage)

# Get full resolution image
log_bias_field = bias_corrector.GetLogBiasFieldAsImage(raw_img_sitk)
corrected_image_full_resolution = raw_img_sitk / sitk.Exp( log_bias_field)

In [ ]:
# Save image
sitk.WriteImage(corrected_image_full_resolution, "E:\\Documents\\N4_Test.nii")